# 第2回 宿題:

## 0. 環境の確認とライブラリの導入

演習から変更なし。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定

演習から変更なし。

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 5,  # 学習データを増やしたのでEpoch数を減らす
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 1024,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab_homework",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/10.学習/AIE2026/lec02/qlora_lab_homework",  # USE_DRIVE=True のときの保存先
    "glossary_sft": "/content/drive/MyDrive/10.学習/AIE2026/lec02/glossary_sft.jsonl",  # 学習データ（用語集の JSONL）
    "eval_json": "/content/drive/MyDrive/10.学習/AIE2026/lec02/eval_glossary_draft.json",  # 評価用の質問と正解（宿題で用意した JSON）
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = True  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

Mounted at /content/drive
{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 5,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 1024,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab_homework",
  "drive_dir": "/content/drive/MyDrive/10.学習/AIE2026/lec02/qlora_lab_homework",
  "glossary_sft": "/content/drive/MyDrive/10.学習/AIE2026/lec02/glossary_sft.jsonl",
  "eval_json": "/content/drive/MyDrive/10.学習/AIE2026/lec02/eval_glossary_draft.json"
}


## 2. モデルの読み込みと学習前評価


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 学習前後で同じ指示を使う。
SYSTEM = "質問された用語の意味を、この分野の文脈で簡潔に説明してください。"

def build_messages(example):
    # 入力は質問だけ。用語集（FAQ）は渡さず、学習した知識だけで答えさせる
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": example["question"]}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [5]:
import pandas as pd
import unicodedata
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

# 評価用の質問と正解（宿題で用意した JSON）。種類ごとに集計する
EVAL_JSON = CONFIG["eval_json"]
with open(EVAL_JSON, encoding="utf-8") as f:
    eval_items = json.load(f)
CATEGORIES = ["用語集にある語", "言い回しを変えた質問", "用語集にない語"]

def normalize(text: str) -> str:
    return "".join(
        unicodedata.normalize("NFKC", text).casefold().split()
    )

def check_answer(text: str, gold: dict) -> bool:
    normalized = normalize(text)
    return all(
        any(normalize(k) in normalized for k in group)
        for group in gold["keyword_groups"]
    )

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''評価用の質問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for item in eval_items:
        raw = generate(model, {"question": item["question"]})   # 入力は質問だけ
        rows.append({"id": item["id"], "category": item["category"], "question": item["question"],
                     "raw": raw, "ok": check_answer(raw, item["gold"])})
    summary = {"label": label, "n": len(rows), "rate": sum(r["ok"] for r in rows) / len(rows)}
    for cat in CATEGORIES:
        cat_rows = [r for r in rows if r["category"] == cat]
        summary[cat] = sum(r["ok"] for r in cat_rows) / len(cat_rows)
    summary["seconds"] = round(time.time() - t0, 1)
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
display(pd.DataFrame([{"ID": r["id"], "種類": r["category"], "質問": r["question"], "学習前の出力": r["raw"], "正解": mark(r["ok"])} for r in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 12, "rate": 0.4166666666666667, "用語集にある語": 0.25, "言い回しを変えた質問": 0.5, "用語集にない語": 0.5, "seconds": 159.5}


,ID,種類,質問,学習前の出力,正解
0,A1,用語集にある語,「AIエンジニアリング」とは何ですか？,**AI エンジニアリング**は、人工知能（AI）の技術、アルゴリズム、およびシステムを設計・構築・運用する専門分野です。\n\nこの分野は、単に「AI を使う」だけでなく、**「AI をどのように設計・構築し、その結果をどのように実装・運用する」**というプロセスを専門的に扱う学問です。\n\n主な特徴は以下の通りです：\n\n* **技術的知見**: 機械学習、深層学習、大規模言語モデル、データサイエンスなどの技術の最新動向を深く理解します。\n* **実装・運用**: 開発された AI システムを、業務プロセスやビジネス課題に合わせて実装し、その結果をどのように評価・分析・改善する（AI エンジニアリングの「実装」という側面）ことが重要です。\n* **ビジネス応用**: 技術的な知識だけでなく、AI をどのようにビジネスに価値を還元するか（例：顧客対応、予測分析、自動化）を明確にします。\n\n一言で言えば、**「AI を設計・構築し、その成果をどのようにビジネスに活用するか」という実践的な知識とスキル集**です。,○
1,A2,用語集にある語,「トークン」とは何ですか？,"**トークン（Token）**とは、ネットワーク通信やデータ処理において、特定のデータ形式やメッセージを識別し、その内容や属性を管理するための**基本単位**です。\n\n特に、**Web 3.0**やブロックチェーン技術において、トークンは以下の意味合いを持ちます：\n\n* **デジタル資産の単位**：仮想通貨（例：Bitcoin, Ethereum）や NFT（非対称なデジタル資産）の基本的な単位として使われます。\n* **取引の単位**：取引（取引手数料や価格）や契約の単位として機能します。\n* **ブロックチェーンの記号**：ブロックチェーン上で「トークン」という記号を指すことで、そのブロックチェーン上の資産や状態を操作・管理する仕組みを指します。\n\n一言で言えば、**「通信や処理の単位」**であり、**「ブロックチェーン上の資産の単位」**という二つの側面から使われます。",×
2,A3,用語集にある語,「LoRA」とは何ですか？,**LoRA**（Low-Rank Adaptation）は、大規模言語モデル（LLM）のトレーニングにおいて、**「少量の知識を、大規模なモデルに追加する」**という手法を指します。\n\n具体的には、モデルの学習プロセスにおいて、学習率や計算コストを大幅に削減するために、**「LoRA（低秩近似）」**と呼ばれる、モデルの構造を維持しつつ、特定の知識やタスクに対してのみ、**少量のパラメータ（約 1〜5% の割合）を調整する**技術です。\n\nこのアプローチの主な利点は以下の通りです：\n* **計算コストの大幅削減**: 大規模モデルを学習する際、パラメータをすべて更新するのは現実的ではありませんが、LoRA では、必要な部分だけを調整することで、学習速度を数倍にします。\n* **知識の可及性**: 大規模モデルに「特定の専門知識」や「特定のタスク」を学習させるため、モデルの知識範囲を拡張し、より深い理解を可能にします。\n* **安定性**: 学習過程でモデルの構造が崩れないため、安定した学習が可能です。\n\n一般的に、LoRA は「LoRA（Low-Rank Adaptation）」と表記され、大規模モデル（如：GPT-4、Claude 3）のトレーニングにおいて最も普及している方法の一つです。,×
3,A4,用語集にある語,「ツール（Tool use / Function calling）」とは何ですか？,"**ツール（Tool use / Function calling）**とは、システムやアプリケーションが外部の**ツール（ツールライブラリや関数）**を使って、特定の動作や処理を実行する行為を指します。\n\nこの文脈では、以下の 2 つの側面が重要です：\n\n1. **ツール自体の活用**：ユーザーが直接操作するのではなく、外部のプログラムや API を呼び出して処理を行うこと。\n2. **関数の呼び出し**：特定の関数（例：`Math.sin()`, `File.open()`, `Database.query()` など）を呼び出して、その関数内部で実行された処理を「ツール用」として処理する。\n\n簡潔に言えば、**「外部の関数やライブラリを使って、システムが処理する行為」**です。",×
4,B1,言い回しを変えた質問,AI Engineeringとは、どのような技術領域ですか？,**AI Engineering**（AI エンジニアリング）は、人工知能（AI）の技術と、その実装・運用を統合した**エンジニアリング**の分野です。\n\n単に「AI を作る」だけでなく、以下の観点から定義されます：\n\n* **技術的基盤**: 大規模言語モデル（LLM）、機械学習アルゴリズム、データ処理、クラウドコンピューティング、セキュリティ、そして**AI 倫理**などの専門知識を駆使します。\n* **実装・運用**: 開発された AI システムを、業務プロセスや組織の文脈に合わせて、**実装・運用・監視・最適化**までまで行うことで、AI が「人間が使う」状態に達します。\n* **目的**: 従来の AI 技術（理論・アルゴリズム）を、**ビジネス・業務の効率化、安全性、自動化**など、人間が直接操作する「人間が使う」状態にまで到達させることを目指します。\n\n一言で言えば、**「AI を設計するエンジニア」と「AI を動かすエンジニア」を統合した総合的な技術領域**です。,○
5,B2,言い回しを変えた質問,LLMでいうtokenは、文章を扱ううえで何を指しますか？,LLM（大規模言語モデル）における**token**は、テキストを扱う際の基本単位です。\n\n具体的には、以下の意味を持ちます：\n\n1. **文字列の単位**：文章を構成する最小単位として、文字列（例えば「Hello World」）を区切ります。\n2. **理解の単位**：モデルが文脈を理解する際に、その文字列を個別に処理する単位です。\n3. **出力の単位**：モデルが回答を生成する際、その文字列を個別に出力します。\n\nつまり、LLM は「文字列を個別に処理する」モデルであり、その基本単位を**token**と呼びます。,○
6,B3,言い回しを変えた質問,Low-Rank Adaptationは、モデルをどのように追加学習する手法ですか？,**Low-Rank Adaptation (LR)** は、モデルを**低次元（低秩）に拡張**して、より高い精度を維持しつつ、**より多くのデータや計算コストで学習**する手法です。\n\n具体的には、モデルの**低秩性（Low-Rankness）**を仮定し、その低秩な部分（低秩部分）を学習対象として、残りの高次元部分（高次元部分）を学習対象として分離して学習します。これにより、学習の効率を向上させ、より多くのデータに対して高い精度を維持できることが特徴です。,×
7,B4,言い回しを変えた質問,LLMのFunction callingでは、モデルとプログラムはそれぞれ何をするのですか？,LLM（大規模言語モデル）の Function Calling において、モデルとプログラムは以下のようになります。\n\n* **プログラム（関数）**：モデルの**入力**を受け取り、**出力**を生成する**タスク**を定義します。\n* **モデル（関数）**：定義されたタスクに対して、**入力を受け取り、出力を生成する**能力を持ちます。\n\nつまり、プログラムは「何をやるか」を定義し、モデルは「そのことをどうやってやるか」を定義します。,×
8,C1,用語集にない語,強化学習って何ですか？,"**強化学習（Reinforcement Learning, RL）**とは、**「人間が直接操作するのではなく、システム（エージェント）が自己で行動を決定し、その行動の結果（報酬）に応じて学習する」**という、**「人間が直接操作する」**という特徴を逆転した学習方法です。\n\n主な特徴は以下の通りです：\n\n* **エージェントの自主性**: 人間が直接操作するのではなく、システムが自己で行動を決定します。\n* **報酬の重要性**: 行動の結果（報酬）が学習の核心となります。\n* **学習の継続性**: 人間が直接操作する学習（例：ゲームのゲームプレイ）では、人間がゲームを終わらせることで学習が終了しますが、強化学習では、人間がゲームを終わらせない限り、学習が継続します。\n\n代表的なアルゴリズムは**Q-Learning**や**DQN**などです。",×
9,C2,用語集にない語,Transformerとは何ですか？,**Transformer** は、**大規模言語モデル（LLM）**や**機械学習**において、**「注意力（Attention）」**を基盤とした**ニューラルネットワーク**の一種です。\n\n特に、**Bert**や**BERT**などの基座モデルにおいて、**「文脈理解」**や**「理解の深さ」**を高めるために開発された技術です

## 3. LoRA 学習（T4 で 4〜5 分）

In [6]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

with open(CONFIG["glossary_sft"], encoding="utf-8") as f:
    glossary_rows = [
        {
            "messages": [
                {"role": "system", "content": SYSTEM}
            ] + json.loads(line)["messages"]
        }
        for line in f
        if line.strip()
    ]
train_ds = Dataset.from_list(glossary_rows)

print("学習データ件数:", len(train_ds))
lengths = [
    len(tok.apply_chat_template(
        row["messages"],
        tokenize=True,
        add_generation_prompt=False,
    ))
    for row in glossary_rows
]

print("最大トークン数:", max(lengths))
print(
    "max_seq_lenを超える件数:",
    sum(n > CONFIG["max_seq_len"] for n in lengths),
)

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=None, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

学習データ件数: 116
最大トークン数: 2
max_seq_lenを超える件数: 0


Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,2.696958
20,1.718183
30,1.332926
40,0.826185
50,0.545581
60,0.417459
70,0.278738


学習時間: 221 秒 | ステップ数: 75 | エポック: 5.0


## 4. 学習後評価（学習前と同じ test・同じ生成設定）

In [7]:
_ = tuned.eval()
after_summary, after_rows = evaluate(tuned, "after")

# 学習前後の要点一致率と評価時間を比較する
compare = (
    pd.DataFrame([before_summary, after_summary])
    .set_index("label")
    [["n", "rate", *CATEGORIES, "seconds"]]
)
display(compare.rename(columns={
    "n": "件数",
    "rate": "全体の要点一致率",
    "seconds": "評価時間（秒）",
}))

# 並び順に依存しないよう、IDで学習前後を対応させる
after_by_id = {r["id"]: r for r in after_rows}
assert {r["id"] for r in before_rows} == set(after_by_id), \
    "学習前後の評価問題のIDが一致していません"

pairs = [(b, after_by_id[b["id"]]) for b in before_rows]

# 自動判定上の退行：○ → ×
regressed = [
    b["question"]
    for b, a in pairs
    if b["ok"] and not a["ok"]
]
print("要点一致が○→×になった質問:", regressed)

# 自動判定上の改善：× → ○
improved = [
    b["question"]
    for b, a in pairs
    if not b["ok"] and a["ok"]
]
print("要点一致が×→○になった質問:", improved)

# 原文を並べ、説明の正確さも確認する
detail = pd.DataFrame([
    {
        "ID": b["id"],
        "種類": b["category"],
        "質問": b["question"],
        "学習前の出力": b["raw"],
        "学習前の要点一致": mark(b["ok"]),
        "学習後の出力": a["raw"],
        "学習後の要点一致": mark(a["ok"]),
    }
    for b, a in pairs
])
display(detail)

{"label": "after", "n": 12, "rate": 0.4166666666666667, "用語集にある語": 0.75, "言い回しを変えた質問": 0.5, "用語集にない語": 0.0, "seconds": 101.8}


,件数,全体の要点一致率,用語集にある語,言い回しを変えた質問,用語集にない語,評価時間（秒）
label,,,,,,
before,12,0.416667,0.25,0.5,0.5,159.5
after,12,0.416667,0.75,0.5,0.0,101.8


要点一致が○→×になった質問: ['「AIエンジニアリング」とは何ですか？', 'Transformerとは何ですか？', 'Dockerとは何ですか？']
要点一致が×→○になった質問: ['「トークン」とは何ですか？', '「LoRA」とは何ですか？', '「ツール（Tool use / Function calling）」とは何ですか？']


,ID,種類,質問,学習前の出力,学習前の要点一致,学習後の出力,学習後の要点一致
0,A1,用語集にある語,「AIエンジニアリング」とは何ですか？,**AI エンジニアリング**は、人工知能（AI）の技術、アルゴリズム、およびシステムを設計・構築・運用する専門分野です。\n\nこの分野は、単に「AI を使う」だけでなく、**「AI をどのように設計・構築し、その結果をどのように実装・運用する」**というプロセスを専門的に扱う学問です。\n\n主な特徴は以下の通りです：\n\n* **技術的知見**: 機械学習、深層学習、大規模言語モデル、データサイエンスなどの技術の最新動向を深く理解します。\n* **実装・運用**: 開発された AI システムを、業務プロセスやビジネス課題に合わせて実装し、その結果をどのように評価・分析・改善する（AI エンジニアリングの「実装」という側面）ことが重要です。\n* **ビジネス応用**: 技術的な知識だけでなく、AI をどのようにビジネスに価値を還元するか（例：顧客対応、予測分析、自動化）を明確にします。\n\n一言で言えば、**「AI を設計・構築し、その成果をどのようにビジネスに活用するか」という実践的な知識とスキル集**です。,○,学習済みの基盤モデル（主にLLM）をツール、ループ、エージェントなどの機能に絞って使い、評価と改善を繰り返す開発スタイル。モデルの選択、ツール設計、評価と回数を数えて進める。本講座では第2回演習で、Geminiの APIをツールとして使い、質問で検索結果をコードとして返す。,×
1,A2,用語集にある語,「トークン」とは何ですか？,"**トークン（Token）**とは、ネットワーク通信やデータ処理において、特定のデータ形式やメッセージを識別し、その内容や属性を管理するための**基本単位**です。\n\n特に、**Web 3.0**やブロックチェーン技術において、トークンは以下の意味合いを持ちます：\n\n* **デジタル資産の単位**：仮想通貨（例：Bitcoin, Ethereum）や NFT（非対称なデジタル資産）の基本的な単位として使われます。\n* **取引の単位**：取引（取引手数料や価格）や契約の単位として機能します。\n* **ブロックチェーンの記号**：ブロックチェーン上で「トークン」という記号を指すことで、そのブロックチェーン上の資産や状態を操作・管理する仕組みを指します。\n\n一言で言えば、**「通信や処理の単位」**であり、**「ブロックチェーン上の資産の単位」**という二つの側面から使われます。",×,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,○
2,A3,用語集にある語,「LoRA」とは何ですか？,**LoRA**（Low-Rank Adaptation）は、大規模言語モデル（LLM）のトレーニングにおいて、**「少量の知識を、大規模なモデルに追加する」**という手法を指します。\n\n具体的には、モデルの学習プロセスにおいて、学習率や計算コストを大幅に削減するために、**「LoRA（低秩近似）」**と呼ばれる、モデルの構造を維持しつつ、特定の知識やタスクに対してのみ、**少量のパラメータ（約 1〜5% の割合）を調整する**技術です。\n\nこのアプローチの主な利点は以下の通りです：\n* **計算コストの大幅削減**: 大規模モデルを学習する際、パラメータをすべて更新するのは現実的ではありませんが、LoRA では、必要な部分だけを調整することで、学習速度を数倍にします。\n* **知識の可及性**: 大規模モデルに「特定の専門知識」や「特定のタスク」を学習させるため、モデルの知識範囲を拡張し、より深い理解を可能にします。\n* **安定性**: 学習過程でモデルの構造が崩れないため、安定した学習が可能です。\n\n一般的に、LoRA は「LoRA（Low-Rank Adaptation）」と表記され、大規模モデル（如：GPT-4、Claude 3）のトレーニングにおいて最も普及している方法の一つです。,×,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習する手法。学習するパラメータが少なく、学習結果（更新行列）を後で再利用できる。,○
3,A4,用語集にある語,「ツール（Tool use / Function calling）」とは何ですか？,"**ツール（Tool use / Function calling）**とは、システムやアプリケーションが外部の**ツール（ツールライブラリや関数）**を使って、特定の動作や処理を実行する行為を指します。\n\nこの文脈では、以下の 2 つの側面が重要です：\n\n1. **ツール自体の活用**：ユーザーが直接操作するのではなく、外部のプログラムや API を呼び出して処理を行うこと。\n2. **関数の呼び出し**：特定の関数（例：`Math.sin()`, `File.open()`, `Database.query()` など）を呼び出して、その関数内部で実行された処理を「ツール用」として処理する。\n\n簡潔に言えば、**「外部の関数やライブラリを使って、システムが処理する行為」**です。",×,モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。,○
4,B1,言い回しを変えた質問,AI Engineeringとは、どのような技術領域ですか？,**AI Engineering**（AI エンジニアリング）は、人工知能（AI）の技術と、その実装・運用を統合した**エンジニアリング**の分野です。\n\n単に「AI を作る」だけでなく、以下の観点から定義されます：\n\n* **技術的基盤**: 大規模言語モデル（LLM）、機械学習アルゴリズム、データ処理、クラウドコンピューティング、セキュリティ、そして**AI 倫理**などの専門知識を駆使します。\n* **実装・運用**: 開発された AI システムを、業務プロセスや組織の文脈に合わせて、**実装・運用・監視・最適化**までまで行うことで、AI が「人間が使う」状態に達します。\n* **目的**: 従来の AI 技術（理論・アルゴリズム）を、**ビジネス・業務の効率化、安全性、自動化**など、人間が直接操作する「人間が使う」状態にまで到達させることを目指します。\n\n一言で言えば、**「AI を設計するエンジニア」と「AI を動かすエンジニア」を統合した総合的な技術領域**です。,○,LLMを設計・構築・評価・運用する全プロセスの技術。基礎知識（機械学習、データ、システム設計）とAIのツールの組み合わせを、ツール呼び出し、評価、回数を数えることで把握する。本講座では「AIエンジニアリングの基礎」で進める。,○
5,B2,言い回しを変えた質問,LLMでいうtokenは、文章を扱ううえで何を指しますか？,LLM（大規模言語モデル）における**token**は、テキストを扱う際の基本単位です。\n\n具体的には、以下の意味を持ちます：\n\n1. **文字列の単位**：文章を構成する最小単位として、文字列（例えば「Hello World」）を区切ります。\n2. **理解の単位**：モデルが文脈を理解する際に、その文字列を個別に処理する単位です。\n3. **出力の単位**：モデルが回答を生成する際、その文字列を個別に出力します。\n\nつまり、LLM は「文字列を個別に処理する」モデルであり、その基本単位を**token**と呼びます。,○,LLMが1つの処理を1つの「トークン」と数える単位。単語や文字数ではなく、モデルが望ましくない断片に文章を割る理由と、その割れ方を数える基準がある。,○
6,B3,言い回しを変えた質問,Low-Rank Adaptationは、モデルをどのように追加学習する手法ですか？,**Low-Rank Adaptation (LR)** は、モデルを**低次元（低秩）に拡張**して、より高い精度を維持しつつ、**より多くのデータや計算コストで学習**する手法です。\n\n具体的には、モデルの**低秩性（Low-Rankness）**を仮定し、その低秩な部分（低秩部分）を学習対象として、残りの高次元部分（高次元部分）を学習対象として分離して学習します。これにより、学習の効率を向上させ、より多くのデータに対して高い精度を維持できることが特徴です。,×,ある次元やある規模のパラメータを低ランク（例：10x10）とみなし、その範囲内の更新を低ランク更新で表し、その範囲をNIterationsで更新し、結果でランクを上げる。更新量とランクの増加を小さくする工夫がある。,×
7,B4,言い回しを変えた質問,LLMのFunction callingでは、モデルとプログラムはそれぞれ何をするのですか？,LLM（大規模言語モデル）の Function Calling において、モデルとプログラムは以下のようになります。\n\n* **プログラム（関数）**：モデルの**入力**を受け取り、**出力**を生

## 5. 比較表と提出

In [8]:
result = {
    "config": {
        k: v for k, v in CONFIG.items()
        if k not in ("work_dir", "drive_dir")
    },
    "versions": {
        m.__name__: m.__version__
        for m in [transformers, peft, trl, torch]
    },
    "method": "LoRAによるSFT（量子化なし）",
    "metric": "キーワード群による要点一致率（グループ内OR・グループ間AND）",
    "before": before_summary,
    "after": after_summary,
    "regressed": regressed,  # 自動判定で○→×になった質問
    "improved": improved,    # 自動判定で×→○になった質問
    "executed_training_myself": True,
}

(WORK / "result.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(result, ensure_ascii=False, indent=2))

if USE_DRIVE:
    import shutil

    drive_dir = Path(CONFIG["drive_dir"])
    drive_dir.mkdir(parents=True, exist_ok=True)

    for name in [
        "result.json",
        "eval_before.json",
        "eval_after.json",
    ]:
        shutil.copy(WORK / name, drive_dir / name)

    print("Drive に保存しました:", drive_dir)

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 5,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 1024,
    "gen": {
      "do_sample": false
    },
    "glossary_sft": "/content/drive/MyDrive/10.学習/AIE2026/lec02/glossary_sft.jsonl",
    "eval_json": "/content/drive/MyDrive/10.学習/AIE2026/lec02/eval_glossary_draft.json"
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRAによるSFT（量子化なし）",
  "metric": "キーワード群による要点一致率（グループ内OR・グループ間AND）",
  "before": {
    "label": "before",
    "n": 12,
    "rate": 0.4166666666666667,
    "用語集にある語": 0.25,
    "言い回しを変えた質問": 0.5,
    "用語集にない語": 0.5,
    "seconds": 159.5
  },
  "after": {
    "label": "after",
    "n": 12,
    "rate": 0.4166666666666667,
    "用語集にある語": 0.75,
    "言い回しを変